# Native CLM v0 — M2 Continual Language Stream

This is the first real continual-language validation. The M1 checkpoint is fixed by SHA-256. Shared substrate/router remain frozen; only Cell operators write. GPU0 runs the protected arm while GPU1 concurrently runs the unsafe control. Formal seeds are 73211/73212/73213 and must not be used for tuning.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from kaggle_secrets import UserSecretsClient

BRANCH = 'codex/native-clm-v0-m2-continual-language'
REPO = Path('/kaggle/working/mini-cells')
CHECKPOINT_DIR = Path('/kaggle/working/native-clm-v0-m1')
CHECKPOINT = CHECKPOINT_DIR / 'final-model.pt'
DATA = Path('/kaggle/working/native-clm-m2-data')
OUT = REPO / 'artifacts/experiments/native-clm-v0-m2-continual-language'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
run(['git', 'checkout', BRANCH])
run(['git', 'reset', '--hard', f'origin/{BRANCH}'])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,lm]'])

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing Kaggle Secret: HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing Kaggle Secret: GITHUB_TOKEN'

import torch
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('torch:', torch.__version__)
print('gpu_count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'gpu[{i}]:', torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Canonical M2 requires two Kaggle GPUs.'


In [ ]:
# Fetch the exact M1 parent from Hugging Face and resolve the current Hub revision.
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m1_checkpoint.py',
    '--repo-id', 'archelabsxyz/native-clm-v0',
    '--filename', 'final-model.pt',
    '--expected-sha256', '91cc66f744c97e50105acbb7cdc328a95cb87a32c49baf5b0d6e462d4d4c4c7f',
    '--output', CHECKPOINT,
])
provenance = json.loads((CHECKPOINT_DIR / 'provenance.json').read_text())
print(json.dumps(provenance, indent=2))


In [ ]:
# Prepare A retention evaluation plus the registered B -> C -> D training stream.
# The dedicated data-prep subprocess works around the Python 3.12 Arrow teardown bug.
run([
    sys.executable, 'scripts/research/prepare_native_clm_v0_m2_data.py',
    '--output-dir', DATA,
])
manifest = json.loads((DATA / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2))


In [ ]:
# Formal M2: for each seed, protected runs on GPU0 and unsafe runs on GPU1 concurrently.
# Return code 2 is a valid registered negative scientific result and must still be published.
cmd = [
    sys.executable, 'scripts/research/run_native_clm_v0_m2.py',
    '--formal',
    '--checkpoint', CHECKPOINT,
    '--data-dir', DATA,
    '--output-dir', OUT,
    '--devices', '0,1',
]
completed = run(cmd, check=False)
print('formal runner return code:', completed.returncode)
assert completed.returncode in (0, 2), 'M2 infrastructure/runtime failure'
decision = json.loads((OUT / 'decision.json').read_text())
print(json.dumps({
    'status': decision['status'],
    'scientific_decision': decision['scientific_decision'],
    'protocol_sha256': decision['protocol_sha256'],
    'completed_seeds': decision['completed_seeds'],
}, indent=2))
for result in decision['seed_results']:
    print('seed', result['seed'], 'pass=', result['pass'],
          'protected_forgetting=', result['protected_mean_forgetting'],
          'unsafe_forgetting=', result['unsafe_mean_forgetting'],
          'advantage=', result['retention_advantage'])


In [ ]:
# Preserve all six formal end-state checkpoints on Hugging Face, then push only lightweight evidence to Git.
# Upload progress is printed file-by-file; .pt files are explicitly excluded from Git.
run([
    sys.executable, 'scripts/research/publish_native_clm_v0_m2.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
    '--checkpoint-provenance', CHECKPOINT_DIR / 'provenance.json',
    '--data-manifest', DATA / 'manifest.json',
    '--hf-repo', 'archelabsxyz/native-clm-v0',
])
print('Published M2 status:', decision['status'])
